# ARM97 PRECT/COSP Run vs IOP Observation

This notebook compares the NCO post-processed ARM97 SCM output in `outputs/arm97_prect_cosp_model_ready.nc` against `outputs/arm97_iop_observation_model_window_nco.nc`.

The observation file is sliced with `ncks -d time,72,1944` so it spans the same absolute window as the model run while preserving the original IOP `time` and `tsec` variables. Observation datetimes are interpreted as `bdate + tsec`.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timedelta
import os
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    return Path("/Users/yunlong/Workshop/SCM-UQ-Workflow")


ROOT = Path(os.environ.get("SCM_UQ_WORKFLOW_ROOT", find_repo_root())).resolve()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".local_cache/matplotlib-cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".local_cache"))

import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date
import plotly.graph_objects as go
from plotly.subplots import make_subplots

MODEL_FILE = Path(
    os.environ.get(
        "ARM97_PRECT_COSP_HISTORY_FILE",
        str(ROOT / "outputs/arm97_prect_cosp_model_ready.nc"),
    )
).expanduser().resolve()

OBSERVATION = Path(
    os.environ.get(
        "ARM97_IOP_FILE",
        str(ROOT / "outputs/arm97_iop_observation_model_window_nco.nc"),
    )
).expanduser().resolve()

OUT_DIR = ROOT / "notebook_outputs" / "arm97_prect_cosp_vs_observation"
FIG_DIR = OUT_DIR / "figures"

assert ROOT.exists(), ROOT
assert MODEL_FILE.exists(), MODEL_FILE
assert OBSERVATION.exists(), OBSERVATION

print("repo root:", ROOT)
print("model file:", MODEL_FILE)
print("observation:", OBSERVATION)
print("output dir:", OUT_DIR)


repo root: /Users/yunlong/Workshop/SCM-UQ-Workflow
model file: /Users/yunlong/Workshop/SCM-UQ-Workflow/outputs/arm97_prect_cosp_model_ready.nc
observation: /Users/yunlong/Workshop/SCM-UQ-Workflow/outputs/arm97_iop_observation_model_window_nco.nc
output dir: /Users/yunlong/Workshop/SCM-UQ-Workflow/notebook_outputs/arm97_prect_cosp_vs_observation


## Variable Mapping

Observation variables are converted to model units before statistics are calculated. `PRECT` is included explicitly and converts observation precipitation from `mm/s` to `m/s`.


In [2]:
@dataclass(frozen=True)
class VarSpec:
    model: str
    obs: str
    units: str
    scale_obs: float = 1.0
    obs_offset: float = 0.0
    description: str = ""


SURFACE_VARS = [
    VarSpec("TREFHT", "Tsair", "K", description="2 m air temperature"),
    VarSpec("TS", "Tg", "K", description="surface/ground temperature"),
    VarSpec("TMQ", "prew", "kg/m2", scale_obs=10.0, description="precipitable water"),
    VarSpec("CLDTOT", "totcld", "1", scale_obs=0.01, description="total cloud fraction"),
    VarSpec("CLDLOW", "lowcld", "1", scale_obs=0.01, description="low cloud fraction"),
    VarSpec("CLDMED", "midcld", "1", scale_obs=0.01, description="mid-level cloud fraction"),
    VarSpec("CLDHGH", "hghcld", "1", scale_obs=0.01, description="high cloud fraction"),
    VarSpec("PS", "Ps", "Pa", description="surface pressure"),
    VarSpec("LHFLX", "lhflx", "W/m2", description="latent heat flux"),
    VarSpec("SHFLX", "shflx", "W/m2", description="sensible heat flux"),
    VarSpec("FSNS", "srfswdn-srfswup", "W/m2", description="surface net shortwave flux"),
    VarSpec("FLNS", "srflwup-srflwdn", "W/m2", description="surface net longwave flux"),
    VarSpec("FSDS", "srfswdn", "W/m2", description="surface downwelling shortwave flux"),
    VarSpec("FLDS", "srflwdn", "W/m2", description="surface downwelling longwave flux"),
    VarSpec("FLUT", "TOA_LWup", "W/m2", description="TOA upwelling longwave flux"),
    VarSpec("U10", "windsrf", "m/s", description="10 m wind speed"),
    VarSpec("PRECT", "Prec", "m/s", scale_obs=0.001, description="total precipitation rate"),
]

pd.DataFrame([vars(v) for v in SURFACE_VARS])


,model,obs,units,scale_obs,obs_offset,description
0,TREFHT,Tsair,K,1.000,0.0,2 m air temperature
1,TS,Tg,K,1.000,0.0,surface/ground temperature
2,TMQ,prew,kg/m2,10.000,0.0,precipitable water
3,CLDTOT,totcld,1,0.010,0.0,total cloud fraction
4,CLDLOW,lowcld,1,0.010,0.0,low cloud fraction
5,CLDMED,midcld,1,0.010,0.0,mid-level cloud fraction
6,CLDHGH,hghcld,1,0.010,0.0,high cloud fraction
7,PS,Ps,Pa,1.000,0.0,surface pressure
8,LHFLX,lhflx,W/m2,1.000,0.0,latent heat flux
9,SHFLX,shflx,W/m2,1.000,0.0,sensible heat flux


## Load And Align Surface Data

The model and observation use different time coordinate conventions, so observation `tsec` is converted to elapsed days from the model start and interpolated onto the model time axis.


In [3]:
def as_series(var):
    data = np.ma.asarray(var[:], dtype=np.float64)
    if data.ndim == 1:
        return data
    axes = tuple(range(1, data.ndim))
    return np.ma.mean(data, axis=axes)


def obs_series(ds, expression: str):
    expression = expression.replace(" ", "")
    if "-" in expression:
        left, right = expression.split("-", 1)
        return as_series(ds.variables[left]) - as_series(ds.variables[right])
    return as_series(ds.variables[expression])


def filled(arr):
    return np.asarray(np.ma.asarray(arr, dtype=np.float64).filled(np.nan), dtype=np.float64)


def load_model_time_axis(ds):
    time = ds.variables["time"]
    days = np.asarray(time[:], dtype=np.float64)
    dates = np.array(
        num2date(days, time.units, getattr(time, "calendar", "standard"), only_use_cftime_datetimes=False)
    )
    return days, dates


def parse_bdate(ds):
    value = int(np.asarray(ds.variables["bdate"][...]).item())
    text = str(value)
    if len(text) == 8:
        return datetime(int(text[:4]), int(text[4:6]), int(text[6:8]))
    if len(text) == 6:
        year = int(text[:2])
        year += 1900 if year >= 70 else 2000
        return datetime(year, int(text[2:4]), int(text[4:6]))
    raise ValueError(f"unsupported bdate value: {value}")


def load_obs_dates(ds):
    base = parse_bdate(ds)
    tsec = np.asarray(ds.variables["tsec"][:], dtype=np.float64)
    return np.array([base + timedelta(seconds=float(x)) for x in tsec], dtype=object)


def obs_relative_days_from_model_origin(obs_dates, origin):
    return np.asarray([(d - origin).total_seconds() / 86400.0 for d in obs_dates], dtype=np.float64)


def interpolate_obs(obs_days, obs_values, target_days):
    finite = np.isfinite(obs_values)
    if finite.sum() < 2:
        return np.full_like(target_days, np.nan, dtype=np.float64)
    return np.interp(target_days, obs_days[finite], obs_values[finite], left=np.nan, right=np.nan)


def stats(model_values, obs_values):
    finite = np.isfinite(model_values) & np.isfinite(obs_values)
    diff = model_values[finite] - obs_values[finite]
    if diff.size == 0:
        return dict(n=0, mean_obs=np.nan, mean_model=np.nan, bias=np.nan, mae=np.nan, rmse=np.nan, max_abs=np.nan)
    return dict(
        n=int(diff.size),
        mean_obs=float(np.mean(obs_values[finite])),
        mean_model=float(np.mean(model_values[finite])),
        bias=float(np.mean(diff)),
        mae=float(np.mean(np.abs(diff))),
        rmse=float(np.sqrt(np.mean(diff * diff))),
        max_abs=float(np.max(np.abs(diff))),
    )


DATA = {}
with Dataset(MODEL_FILE) as model, Dataset(OBSERVATION) as obs:
    model_days, model_dates = load_model_time_axis(model)
    origin = model_dates[0] - timedelta(days=float(model_days[0]))
    obs_dates = load_obs_dates(obs)
    obs_days = obs_relative_days_from_model_origin(obs_dates, origin)

    for spec in SURFACE_VARS:
        obs_names = [name.strip() for name in spec.obs.replace("-", ",").split(",")]
        if spec.model not in model.variables or any(name not in obs.variables for name in obs_names):
            continue
        model_values = filled(as_series(model.variables[spec.model]))
        obs_native = filled(obs_series(obs, spec.obs)) * spec.scale_obs + spec.obs_offset
        obs_at_model = interpolate_obs(obs_days, obs_native, model_days)
        diff = model_values - obs_at_model
        DATA[spec.model] = dict(
            spec=spec,
            model_dates=model_dates,
            obs_dates=obs_dates,
            model_days=model_days,
            obs_days=obs_days,
            model_values=model_values,
            obs_native=obs_native,
            obs_at_model=obs_at_model,
            diff=diff,
            stats=stats(model_values, obs_at_model),
        )

summary = pd.DataFrame([
    {
        "variable": name,
        "observation": d["spec"].obs,
        "description": d["spec"].description,
        "units": d["spec"].units,
        "n": d["stats"]["n"],
        "mean_obs": d["stats"]["mean_obs"],
        "mean_model": d["stats"]["mean_model"],
        "bias_model_minus_obs": d["stats"]["bias"],
        "mae": d["stats"]["mae"],
        "rmse": d["stats"]["rmse"],
        "max_abs_error": d["stats"]["max_abs"],
    }
    for name, d in DATA.items()
]).sort_values("rmse", ascending=False)

print(f"Loaded {len(DATA)} surface variables.")
summary


Loaded 17 surface variables.


,variable,observation,description,units,n,mean_obs,mean_model,bias_model_minus_obs,mae,rmse,max_abs_error
12,FSDS,srfswdn,surface downwelling shortwave flux,W/m2,1248,2.823433e+02,2.324231e+02,-4.992015e+01,8.241398e+01,1.304188e+02,5.453253e+02
10,FSNS,srfswdn-srfswup,surface net shortwave flux,W/m2,1248,2.272567e+02,1.997924e+02,-2.746425e+01,6.868999e+01,1.072572e+02,4.338545e+02
14,FLUT,TOA_LWup,TOA upwelling longwave flux,W/m2,1248,2.640322e+02,1.874872e+02,-7.654503e+01,7.756161e+01,9.316196e+01,1.990635e+02
11,FLNS,srflwup-srflwdn,surface net longwave flux,W/m2,1248,6.337264e+01,3.382397e+01,-2.954867e+01,3.054507e+01,3.763966e+01,1.001726e+02
13,FLDS,srflwdn,surface downwelling longwave flux,W/m2,1248,3.932977e+02,4.192935e+02,2.599575e+01,2.659220e+01,3.157107e+01,7.447188e+01
7,PS,Ps,surface pressure,Pa,1248,9.686515e+04,9.686423e+04,-9.134080e-01,1.738164e+01,2.189918e+01,7.447907e+01
2,TMQ,prew,precipitable water,kg/m2,1248,3.631311e+01,4.326576e+01,6.952653e+00,7.737019e+00,8.883605e+00,1.714431e+01
8,LHFLX,lhflx,latent heat flux,W/m2,1248,1.129977e+02,1.129664e+02,-3.136420e-02,3.964817e+00,6.571690e+00,2.158137e+01
9,SHFLX,shflx,sensible heat flux,W/m2,1248,3.673418e+01,3.672063e+01,-1.354659e-02,2.070002e+00,3.259283e+00,1.265830e+01
0,TREFHT,Tsair,2 m air temperature,K,1248,2.991863e+02,3.011402e+02,1.953953e+00,2.345290e+00,2.877355e+00,9.611274e+00


## Interactive Surface Comparison

Use the dropdown to switch variables. The top panel overlays model and observation; the bottom panel shows `model - observation` on the model time axis.


In [4]:
def title_for(name, d):
    s = d["stats"]
    return (
        f"{name}: ARM97 baseline vs observation"
        f"<br><sup>{d['spec'].description} | obs={d['spec'].obs} | "
        f"RMSE={s['rmse']:.3g}, bias={s['bias']:.3g} {d['spec'].units}, n={s['n']}</sup>"
    )

names = list(DATA)
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    row_heights=[0.68, 0.32],
)

buttons = []
TRACES_PER_VARIABLE = 3
for i, name in enumerate(names):
    d = DATA[name]
    visible = i == 0
    fig.add_trace(
        go.Scatter(
            x=d["model_dates"],
            y=d["model_values"],
            mode="lines",
            name="model",
            legendrank=2,
            line=dict(color="#1261A6", width=2.0),
            visible=visible,
            hovertemplate="%{x}<br>model=%{y:.4g}<extra></extra>",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=d["obs_dates"],
            y=d["obs_native"],
            mode="lines+markers",
            name="observation",
            legendrank=1,
            line=dict(color="black", width=2.8),
            marker=dict(color="black", size=3, opacity=0.85),
            visible=visible,
            hovertemplate="%{x}<br>obs=%{y:.4g}<extra></extra>",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=d["model_dates"],
            y=d["diff"],
            mode="lines",
            name="model - observation",
            legendrank=3,
            line=dict(color="#C2410C", width=1.5),
            visible=visible,
            hovertemplate="%{x}<br>model - obs=%{y:.4g}<extra></extra>",
        ),
        row=2,
        col=1,
    )

    mask = [False] * (len(names) * TRACES_PER_VARIABLE)
    mask[i * TRACES_PER_VARIABLE : i * TRACES_PER_VARIABLE + TRACES_PER_VARIABLE] = [True] * TRACES_PER_VARIABLE
    buttons.append(
        dict(
            label=f"{name}: {d['spec'].description}",
            method="update",
            args=[
                {"visible": mask},
                {
                    "title.text": title_for(name, d),
                    "yaxis.title.text": f"{name} ({d['spec'].units})",
                    "yaxis2.title.text": f"model - observation ({d['spec'].units})",
                },
            ],
        )
    )

first = DATA[names[0]]
fig.update_layout(
    title=dict(text=title_for(names[0], first), x=0.01, xanchor="left"),
    template="plotly_white",
    height=740,
    width=1120,
    hovermode="x unified",
    margin=dict(l=80, r=40, t=120, b=70),
    legend=dict(orientation="h", yanchor="bottom", y=1.005, xanchor="left", x=0),
    updatemenus=[
        dict(type="dropdown", x=1.0, y=1.13, xanchor="right", yanchor="top", buttons=buttons, showactive=True)
    ],
)
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_xaxes(title="Time", row=2, col=1)
fig.update_yaxes(title=f"{names[0]} ({first['spec'].units})", row=1, col=1)
fig.update_yaxes(
    title=f"model - observation ({first['spec'].units})",
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor="black",
    row=2,
    col=1,
)
fig.show()


## Pressure-Level Profile Comparison

For profile variables, model hybrid levels are converted to pressure with `hyam * P0 + hybm * PS(time)`, then interpolated to the selected observation pressure level.


In [5]:
@dataclass(frozen=True)
class ProfileSpec:
    model: str
    obs: str
    units: str
    description: str = ""


PROFILE_VARS = [
    ProfileSpec("T", "T", "K", "temperature"),
    ProfileSpec("Q", "q", "kg/kg", "specific humidity"),
    ProfileSpec("U", "u", "m/s", "zonal wind"),
    ProfileSpec("V", "v", "m/s", "meridional wind"),
    ProfileSpec("OMEGA", "omega", "Pa/s", "pressure vertical velocity"),
    ProfileSpec("RELHUM", "rh", "%", "relative humidity"),
]


def interp_model_matrix_to_pressure(values, pressure, target_pressure):
    values = np.asarray(values, dtype=np.float64)
    pressure = np.asarray(pressure, dtype=np.float64)
    idx = np.sum(pressure < target_pressure, axis=1)
    valid = (idx > 0) & (idx < pressure.shape[1])
    out = np.full(values.shape[0], np.nan, dtype=np.float64)
    if not valid.any():
        return out

    rows = np.arange(values.shape[0])[valid]
    upper = idx[valid]
    lower = upper - 1
    p0 = pressure[rows, lower]
    p1 = pressure[rows, upper]
    v0 = values[rows, lower]
    v1 = values[rows, upper]
    ok = np.isfinite(p0) & np.isfinite(p1) & np.isfinite(v0) & np.isfinite(v1) & (p1 != p0)
    interp = np.full(rows.shape[0], np.nan, dtype=np.float64)
    interp[ok] = v0[ok] + (target_pressure - p0[ok]) * (v1[ok] - v0[ok]) / (p1[ok] - p0[ok])
    out[rows] = interp
    return out


def model_pressure(ds):
    p0 = float(np.asarray(ds.variables["P0"][...]))
    hyam = np.asarray(ds.variables["hyam"][:], dtype=np.float64)
    hybm = np.asarray(ds.variables["hybm"][:], dtype=np.float64)
    ps = np.asarray(ds.variables["PS"][:], dtype=np.float64).squeeze()
    return hyam[None, :] * p0 + hybm[None, :] * ps[:, None]


def load_profile_data():
    profile = {}
    with Dataset(MODEL_FILE) as model, Dataset(OBSERVATION) as obs:
        model_days, model_dates = load_model_time_axis(model)
        origin = model_dates[0] - timedelta(days=float(model_days[0]))
        obs_dates = load_obs_dates(obs)
        obs_days = obs_relative_days_from_model_origin(obs_dates, origin)

        obs_levels_pa = np.asarray(obs.variables["lev"][:], dtype=np.float64)
        pressure = model_pressure(model)

        for spec in PROFILE_VARS:
            if spec.model not in model.variables or spec.obs not in obs.variables:
                continue
            model_values = np.asarray(model.variables[spec.model][:], dtype=np.float64).squeeze()
            obs_values = np.asarray(obs.variables[spec.obs][:], dtype=np.float64).squeeze()
            if obs_values.ndim > 2:
                obs_values = np.nanmean(obs_values, axis=tuple(range(2, obs_values.ndim)))

            level_data = {}
            for level_index, target_pressure in enumerate(obs_levels_pa):
                model_at_level = interp_model_matrix_to_pressure(model_values, pressure, target_pressure)
                obs_native = obs_values[:, level_index]
                obs_at_model = interpolate_obs(obs_days, obs_native, model_days)
                level_data[float(target_pressure)] = dict(
                    model_at_level=model_at_level,
                    obs_native=obs_native,
                    obs_at_model=obs_at_model,
                    diff=model_at_level - obs_at_model,
                    stats=stats(model_at_level, obs_at_model),
                )

            profile[spec.model] = dict(
                spec=spec,
                model_dates=model_dates,
                obs_dates=obs_dates,
                obs_levels_pa=obs_levels_pa,
                level_data=level_data,
            )
    return profile


PROFILE_DATA = load_profile_data()
print(f"Loaded {len(PROFILE_DATA)} profile variables.")


Loaded 6 profile variables.


In [6]:
try:
    import ipywidgets as widgets
    import matplotlib.dates as mdates
    import matplotlib.pyplot as plt
    from IPython.display import display

    profile_names = list(PROFILE_DATA)
    pressure_options = [
        (f"{p / 100:.0f} hPa", float(p))
        for p in next(iter(PROFILE_DATA.values()))["obs_levels_pa"]
        if float(p) < 96500.0
    ]
    pressure_values = [value for _, value in pressure_options]
    default_pressure = min(pressure_values, key=lambda p: abs(p - 50000.0))

    profile_var = widgets.Dropdown(
        options=[(f"{name}: {PROFILE_DATA[name]['spec'].description}", name) for name in profile_names],
        value="T" if "T" in profile_names else profile_names[0],
        description="variable",
        layout=widgets.Layout(width="430px"),
    )
    pressure_level = widgets.SelectionSlider(
        options=pressure_options,
        value=default_pressure,
        description="level",
        continuous_update=False,
        readout=True,
        layout=widgets.Layout(width="620px"),
        style={"description_width": "50px"},
    )

    def draw_profile(name, level_pa):
        level_pa = float(level_pa)
        item = PROFILE_DATA[name]
        spec = item["spec"]
        d = item["level_data"][level_pa]
        s = d["stats"]

        fig, axes = plt.subplots(
            2,
            1,
            figsize=(12, 7),
            sharex=True,
            gridspec_kw={"height_ratios": [2.1, 1.0], "hspace": 0.08},
        )
        axes[0].plot(item["model_dates"], d["model_at_level"], color="#1261A6", lw=2.0, label="model", zorder=2)
        axes[0].plot(item["obs_dates"], d["obs_native"], color="black", lw=1.8, alpha=0.95, label="observation", zorder=4)
        axes[0].set_ylabel(f"{name} ({spec.units})")
        axes[0].legend(loc="best", frameon=False)
        axes[0].grid(True, alpha=0.25)

        axes[1].plot(item["model_dates"], d["diff"], color="#C2410C", lw=1.3, label="model - observation")
        axes[1].axhline(0, color="black", lw=0.8)
        axes[1].set_ylabel(f"diff ({spec.units})")
        axes[1].set_xlabel("Time")
        axes[1].legend(loc="best", frameon=False)
        axes[1].grid(True, alpha=0.25)

        axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=4))
        axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        fig.suptitle(
            f"{name} at {level_pa / 100:.0f} hPa: PRECT/COSP run vs observation\n"
            f"{spec.description} | RMSE={s['rmse']:.3g}, bias={s['bias']:.3g} {spec.units}, n={s['n']}",
            x=0.02,
            ha="left",
            y=0.98,
        )
        fig.autofmt_xdate(rotation=0)
        fig.subplots_adjust(top=0.86, left=0.08, right=0.98, bottom=0.10)
        display(fig)
        plt.close(fig)

    out = widgets.interactive_output(draw_profile, {"name": profile_var, "level_pa": pressure_level})
    display(widgets.VBox([profile_var, pressure_level]), out)
except Exception as exc:
    print("ipywidgets profile controls are unavailable in this kernel.")
    print(repr(exc))


Output()